In [251]:
import sympy as sym
import numpy as np
import dill
import time
import random
from scipy import constants
from IPython.display import Math, display

sym.init_printing(use_unicode=True)

print(np.__version__)

2.5.2


# Intro

## Preliminaries

$$
\begin{align}
q\{\phi \} &\triangleq \text{Exp}(\phi) = \begin{bmatrix} \cos(\frac{||\phi||}{2}) \\ \frac{\phi}{||\phi||} \sin(\frac{||\phi||}{2}) \end{bmatrix} \approx \begin{bmatrix} 1 \\ \frac{\phi}{2} \end{bmatrix} \text{ for small angles.} \\
\\
[\omega]_\times &\triangleq \begin{bmatrix}
0 & -\omega_z & \omega_y \\
\omega_z & 0 & -\omega_x \\
-\omega_y & \omega_x & 0
\end{bmatrix} \\
\\
\mathbf{R}^{\{q\}} &= \begin{bmatrix} 1 - 2(q_y^2 + q_z^2) & 2(q_x q_y - q_z q_w) & 2(q_x q_z + q_y q_w) \\ 2(q_x q_y + q_z q_w) & 1 - 2(q_x^2 + q_z^2) & 2(q_y q_z - q_x q_w) \\ 2(q_x q_z - q_y q_w) & 2(q_y q_z + q_x q_w) & 1 - 2(q_x^2 + q_y^2) \end{bmatrix}
\end{align}
$$

In [252]:
def quat_from_axis_angle(Phi, small_angle=True):
    if small_angle:
        return sym.Matrix([1, *(0.5 * Phi)])
    
    norm = sym.sqrt(Phi[0]**2 + Phi[1]**2 + Phi[2]**2)
    return sym.Matrix([
        sym.cos(0.5 * norm),
        *(Phi * (1 / norm) * sym.sin(0.5 * norm))
    ])

def skew_symmetric_matrix(omega):
    omega_x, omega_y, omega_z = omega[0], omega[1], omega[2]
    return sym.Matrix([
        [0, -omega_z, omega_y],
        [omega_z, 0, -omega_x],
        [-omega_y, omega_x, 0]
    ])

# rotation matrix from quaternion
def R(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    return sym.Matrix([
        [1 - 2*(qy**2 + qz**2), 2*(qx*qy - qz*qw),     2*(qx*qz + qy*qw)],
        [2*(qx*qy + qz*qw),     1 - 2*(qx**2 + qz**2), 2*(qy*qz - qx*qw)],
        [2*(qx*qz - qy*qw),     2*(qy*qz + qx*qw),     1 - 2*(qx**2 + qy**2)]
    ])

R(sym.Matrix(sym.symbols(r"q_w q_x q_y q_z", real=True)))

⎡       2        2                                                  ⎤
⎢- 2⋅q_y  - 2⋅q_z  + 1  -2⋅q_w⋅q_z + 2⋅qₓ⋅q_y  2⋅q_w⋅q_y + 2⋅qₓ⋅q_z ⎥
⎢                                                                   ⎥
⎢                             2        2                            ⎥
⎢2⋅q_w⋅q_z + 2⋅qₓ⋅q_y   - 2⋅qₓ  - 2⋅q_z  + 1   -2⋅q_w⋅qₓ + 2⋅q_y⋅q_z⎥
⎢                                                                   ⎥
⎢                                                    2        2     ⎥
⎣-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z  2⋅q_w⋅qₓ + 2⋅q_y⋅q_z   - 2⋅qₓ  - 2⋅q_y  + 1 ⎦

In [253]:
def quat_inv(q):
    return sym.Matrix([q[0], -q[1], -q[2], -q[3]])
    
def quat_norm(q):
    return q / sym.sqrt(q[0]**2 + q[1]**2 + q[2]**2 + q[3]**2)


def left_quat_matrix(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    
    return sym.Matrix([
        [qw, -qx, -qy, -qz], 
        [qx,  qw, -qz,  qy], 
        [qy,  qz,  qw, -qx], 
        [qz, -qy,  qx,  qw]  
    ])

## Fixed Values

For each cycle, we calculate $\Delta t$ in code and we assume a fixed gravity acceleration force. For the offset between IMU and camera, we define translation and rotation.

\begin{align*}
\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\frac{102°}{2}\right) & \displaystyle \sin\left(-\frac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top \\
\end{align*}

In [254]:
dt = sym.Symbol(r'\Delta t', real=True, positive=True)
subs_dt = {dt: 0.01} # approx 100 Hz sampling rate (not actually fixed in the system but fixed per discrete time step)

gravity = sym.Symbol(r'g', real=True, positive=True)
subs_gravity = {gravity: constants.g}

# \Delta p_{I \to c}
IMU_cam_x_translation = 0
IMU_cam_y_translation = sym.Symbol(r'\Delta{p}_y', negative=True)
IMU_cam_z_translation = sym.Symbol(r'\Delta{p}_z', negative=True)
IMU_cam_translation = sym.Matrix([IMU_cam_x_translation, IMU_cam_y_translation, IMU_cam_z_translation])

subs_IMU_cam_translation = {
    IMU_cam_y_translation: -0.0013,
    IMU_cam_z_translation: -0.00662
}

# \Delta q_{I \to c}
IMU_cam_w_rotation = sym.Symbol(r'\Delta{q}_w', real=True)
IMU_cam_x_rotation = sym.Symbol(r'\Delta{q}_x', real=True)
IMU_cam_y_rotation = 0
IMU_cam_z_rotation = 0
IMU_cam_rotation = sym.Matrix([IMU_cam_w_rotation, IMU_cam_x_rotation, IMU_cam_y_rotation, IMU_cam_z_rotation])

Phi = 102 * (constants.pi / 180)
subs_IMU_cam_rotation = {
    IMU_cam_w_rotation: sym.cos(- Phi / 2),
    IMU_cam_x_rotation: sym.sin(- Phi / 2),
}

subs_fixed_values = subs_dt | subs_gravity | subs_IMU_cam_rotation | subs_IMU_cam_translation

# how substitution works later, .subs() does not overwrite my sympy expressions
dt.subs(subs_dt), gravity.subs(subs_gravity), IMU_cam_translation.subs(subs_IMU_cam_translation), IMU_cam_rotation.subs(subs_IMU_cam_rotation)

⎛                           ⎡0.629320391049838 ⎤⎞
⎜               ⎡   0    ⎤  ⎢                  ⎥⎟
⎜               ⎢        ⎥  ⎢-0.777145961456971⎥⎟
⎜0.01, 9.80665, ⎢-0.0013 ⎥, ⎢                  ⎥⎟
⎜               ⎢        ⎥  ⎢        0         ⎥⎟
⎜               ⎣-0.00662⎦  ⎢                  ⎥⎟
⎝                           ⎣        0         ⎦⎠

---
# Input

\begin{align*}
&\text{Accelerometer input: } &\tilde a && &\text{noise: } &\tilde a_n \\
&\text{Gyroscope input: } &\tilde \omega && &\text{noise: } &\tilde \omega_n\\
\end{align*}

In [255]:
a_tilde = sym.Matrix(sym.symbols(r"\tilde{a}_x \tilde{a}_y \tilde{a}_z", real=True))
omega_tilde = sym.Matrix(sym.symbols(r"\tilde{\omega}_x \tilde{\omega}_y \tilde{\omega}_z", real=True))
u_tilde = sym.Matrix.vstack(a_tilde, omega_tilde)

a_tilde_n = sym.Matrix(sym.symbols(r"\tilde{a}_{n\,x} \tilde{a}_{n\,y} \tilde{a}_{n\,z}", real=True))
omega_tilde_n = sym.Matrix(sym.symbols(r"\tilde{\omega}_{n\,x} \tilde{\omega}_{n\,y} \tilde{\omega}_{n\,z}", real=True))
u_tilde_n = sym.Matrix.vstack(a_tilde_n, omega_tilde_n)

u_tilde.T, u_tilde_n.T

([\tilde{a}ₓ  \tilde{a}_y  \tilde{a}_z  \tilde{\omega}ₓ  \tilde{\omega}_y  \ti ↪

↪ lde{\omega}_z], [\tilde{a}_{n,x}  \tilde{a}_{n,y}  \tilde{a}_{n,z}  \tilde{\ ↪

↪ omega}_{n,x}  \tilde{\omega}_{n,y}  \tilde{\omega}_{n,z}])

---
# Nominal State

## IMU

\begin{align*}
x_\text{IMU} = \begin{bmatrix} p & v & q & a_b & \omega_b \end{bmatrix}^\top \qquad \qquad x \in SO(3) \times \left(\mathbb{R}^3\right)^4
\end{align*}

In [256]:
p = sym.Matrix(sym.symbols(r"p_x p_y p_z", real=True))                                     # position
v = sym.Matrix(sym.symbols(r"v_x v_y v_z", real=True))                                     # velocity
q = sym.Matrix(sym.symbols(r"q_w q_x q_y q_z", real=True))                                 # orientation quaternion
a_b = sym.Matrix(sym.symbols(r"a_{b\,x} a_{b\,y} a_{b\,z}", real=True))                    # accelerometer bias
omega_b = sym.Matrix(sym.symbols(r"\omega_{b\,x} \omega_{b\,y} \omega_{b\,z}", real=True)) # gyroscope bias

x_IMU = sym.Matrix.vstack(p, v, q, a_b, omega_b)
x_IMU.T

[pₓ  p_y  p_z  vₓ  v_y  v_z  q_w  qₓ  q_y  q_z  a_{b,x}  a_{b,y}  a_{b,z}  \om ↪

↪ ega_{b,x}  \omega_{b,y}  \omega_{b,z}]

### Kinematics

\begin{align*}
p &\leftarrow p + v\Delta t + \frac{1}{2}( \mathbf{R}^{\{q\}} (\tilde a - a_b) + g)\Delta t^2 \qquad & a_b &\leftarrow a_b \\
\text{Discrete Case} \qquad \qquad v &\leftarrow v + (\mathbf{R}^{\{q\}}(\tilde a - a_b) + g)\Delta t & \omega_b &\leftarrow \omega_b \\
q &\leftarrow q \otimes q\{(\tilde \omega - \omega_b) \Delta t\} \\
\end{align*}

In [257]:
f_p = p + v * dt + 0.5 * (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt**2
f_v = v + (R(q) * (a_tilde - a_b) + sym.Matrix([0, 0, gravity])) * dt
f_q = left_quat_matrix(q) * quat_from_axis_angle((omega_tilde - omega_b) * dt)
f_a_b = a_b
f_omega_b = omega_b

f_nominal_state_IMU = sym.Matrix.vstack(f_p, f_v, f_q, f_a_b, f_omega_b)
# f_nominal_state_IMU

## Speakers

\begin{align*}
x_s = \begin{bmatrix} p_i & v_i & q_i \end{bmatrix}^\top \qquad \qquad x_s \in SO(3) \times \left(\mathbb{R}^3\right)^2
\end{align*}

In [258]:
p_i = sym.Matrix(sym.symbols(r"p_{i\,x} p_{i\,y} p_{i\,z}", real=True))
v_i = sym.Matrix(sym.symbols(r"v_{i\,x} v_{i\,y} v_{i\,z}", real=True))
q_i = sym.Matrix(sym.symbols(r"q_{i\,w} q_{i\,x} q_{i\,y} q_{i\,z}", real=True))

x_speaker = sym.Matrix.vstack(p_i, v_i, q_i)
x_speaker.T

[p_{i,x}  p_{i,y}  p_{i,z}  v_{i,x}  v_{i,y}  v_{i,z}  q_{i,w}  q_{i,x}  q_{i, ↪

↪ y}  q_{i,z}]

### Kinematics

\begin{align*}
p_i &\leftarrow p_i + v_i \Delta t \\
\text{Discrete Case} \qquad \qquad v_i &\leftarrow v_i \\
q_i &\leftarrow q_i \\
\end{align*}

In [259]:
f_p_i = p_i + v_i * dt
f_v_i = v_i
f_q_i = q_i

f_nominal_state_speaker = sym.Matrix.vstack(f_p_i, f_v_i, f_q_i)
# f_nominal_state_speaker

## Combined State

\begin{align*}
x = \begin{bmatrix} p & v & q & a_b & \omega_b & p_i & v_i & q_i & p_{i + 1} & v_{i + 1} & q_{i + 1} & \cdots \end{bmatrix}^\top
\end{align*}

In [260]:
x = sym.Matrix.vstack(x_IMU, x_speaker)
f_nominal_state = sym.Matrix.vstack(f_nominal_state_IMU, f_nominal_state_speaker)

f_nominal_state

⎡            2 ⎛                           ⎛       2        2    ⎞             ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ + 0.5⋅(\til ↪
⎢                                                                              ↪
⎢            2 ⎛                                                               ↪
⎢    \Delta t ⋅⎝0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(2⋅q_w⋅q_z + 2⋅qₓ⋅q_y) + 0.5⋅(\tild ↪
⎢                                                                              ↪
⎢        2 ⎛                                                                   ↪
⎢\Delta t ⋅⎝0.5⋅g + 0.5⋅(\tilde{a}ₓ - a_{b,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ⋅q_z) + 0.5⋅( ↪
⎢                                                                              ↪
⎢                           ⎛                       ⎛       2        2    ⎞    ↪
⎢                  \Delta t⋅⎝(\tilde{a}ₓ - a_{b,x})⋅⎝- 2⋅q_y  - 2⋅q_z  + 1⎠ +  ↪
⎢                                                                              ↪
⎢                           

---
# Error State

## IMU

\begin{align*}
\delta x_\text{IMU} = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b \end{bmatrix}^\top \qquad \qquad x \in \left(\mathbb{R}^3\right)^5
\end{align*}

In [261]:
delta_p = sym.Matrix(sym.symbols(r"\delta{p}_x \delta{p}_y \delta{p}_z", real=True))
delta_v = sym.Matrix(sym.symbols(r"\delta{v}_x \delta{v}_y \delta{v}_z", real=True))
delta_theta = sym.Matrix(sym.symbols(r"\delta{\theta}_x \delta{\theta}_y \delta{\theta}_z", real=True))
delta_a_b = sym.Matrix(sym.symbols(r"\delta{a}_{b\,x} \delta{a}_{b\,y} \delta{a}_{b\,z}", real=True))
delta_omega_b = sym.Matrix(sym.symbols(r"\delta{\omega}_{b\,x} \delta{\omega}_{b\,y} \delta{\omega}_{b\,z}", real=True))

delta_x_IMU = sym.Matrix.vstack(delta_p, delta_v, delta_theta, delta_a_b, delta_omega_b)

# IMU Process Noise
delta_v_n = sym.Matrix(sym.symbols(r"\delta{v}_{n\,x} \delta{v}_{n\,y} \delta{v}_{n\,z}", real=True))
delta_theta_n = sym.Matrix(sym.symbols(r"\delta{\theta}_{n\,x} \delta{\theta}_{n\,y} \delta{\theta}_{n\,z}", real=True))
delta_a_n = sym.Matrix(sym.symbols(r"\delta{a}_{n\,x} \delta{a}_{n\,y} \delta{a}_{n\,z}", real=True))
delta_omega_n = sym.Matrix(sym.symbols(r"\delta{\omega}_{n\,x} \delta{\omega}_{n\,y} \delta{\omega}_{n\,z}", real=True))

delta_x_n_IMU = sym.Matrix.vstack(delta_v_n, delta_theta_n, delta_a_n, delta_omega_n)

delta_x_IMU.T, delta_x_n_IMU.T

([\delta{p}ₓ  \delta{p}_y  \delta{p}_z  \delta{v}ₓ  \delta{v}_y  \delta{v}_z   ↪

↪ \delta{\theta}ₓ  \delta{\theta}_y  \delta{\theta}_z  \delta{a}_{b,x}  \delta ↪

↪ {a}_{b,y}  \delta{a}_{b,z}  \delta{\omega}_{b,x}  \delta{\omega}_{b,y}  \del ↪

↪ ta{\omega}_{b,z}], [\delta{v}_{n,x}  \delta{v}_{n,y}  \delta{v}_{n,z}  \delt ↪

↪ a{\theta}_{n,x}  \delta{\theta}_{n,y}  \delta{\theta}_{n,z}  \delta{a}_{n,x} ↪

↪   \delta{a}_{n,y}  \delta{a}_{n,z}  \delta{\omega}_{n,x}  \delta{\omega}_{n, ↪

↪ y}  \delta{\omega}_{n,z}])

### Kinematics

\begin{align*}
\delta p &\leftarrow \delta p + \delta v\Delta t & \delta a_b &\leftarrow \delta a_b + \delta a_n \\
\text{Discrete Case} \qquad \qquad \delta v &\leftarrow \delta v + (-\mathbf{R}^{\{q\}}[\tilde a - a_b]_\times \delta \theta - \mathbf{R}^{\{q\}} \delta a_b)\Delta t + \delta v_{n} \qquad & \delta \omega_b &\leftarrow \delta \omega_b + \delta \omega_n \\
\delta \theta &\leftarrow (\mathbf{R}^{\{(\tilde \omega - \omega_b) \Delta t\}})^\top \delta \theta - \delta \omega_b \Delta t + \delta \theta_{n}
\end{align*}

In [262]:
f_delta_p = delta_p + delta_v * dt
f_delta_v = delta_v + (- R(q) * skew_symmetric_matrix(a_tilde - a_b) * delta_theta \
    - R(q) * delta_a_b) * dt + delta_v_n
f_delta_theta = R(quat_from_axis_angle((omega_tilde - omega_b) * dt)).T * delta_theta \
    - delta_omega_b * dt + delta_theta_n
f_delta_a_b = delta_a_b + delta_a_n
f_delta_omega_b = delta_omega_b + delta_omega_n

f_error_state_IMU = sym.Matrix.vstack(f_delta_p, f_delta_v, f_delta_theta, f_delta_a_b, f_delta_omega_b)
# f_error_state_IMU

## Speakers

\begin{align*}
\delta x_s = \begin{bmatrix} \delta p_i & \delta v_i & \delta \theta_i \end{bmatrix}^\top \qquad \qquad x_s \in \left(\mathbb{R}^3\right)^3
\end{align*}

In [263]:
delta_p_i = sym.Matrix(sym.symbols(r"\delta{p}_{i\,x} \delta{p}_{i\,y} \delta{p}_{i\,z}", real=True))
delta_v_i = sym.Matrix(sym.symbols(r"\delta{v}_{i\,x} \delta{v}_{i\,y} \delta{v}_{i\,z}", real=True))
delta_theta_i = sym.Matrix(sym.symbols(r"\delta{\theta}_{i\,x} \delta{\theta}_{i\,y} \delta{\theta}_{i\,z}", real=True))

delta_x_speaker = sym.Matrix.vstack(delta_p_i, delta_v_i, delta_theta_i)

# Speaker Process Noise
delta_v_i_n = sym.Matrix(sym.symbols(r"\delta{v}_{in\,x} \delta{v}_{in\,y} \delta{v}_{in\,z}", real=True))
delta_theta_i_n = sym.Matrix(sym.symbols(r"\delta{\theta}_{in\,x} \delta{\theta}_{in\,y} \delta{\theta}_{in\,z}", real=True))

delta_x_n_speaker = sym.Matrix.vstack(delta_v_i_n, delta_theta_i_n)

delta_x_speaker.T, delta_x_n_speaker.T


([\delta{p}_{i,x}  \delta{p}_{i,y}  \delta{p}_{i,z}  \delta{v}_{i,x}  \delta{v ↪

↪ }_{i,y}  \delta{v}_{i,z}  \delta{\theta}_{i,x}  \delta{\theta}_{i,y}  \delta ↪

↪ {\theta}_{i,z}], [\delta{v}_{in,x}  \delta{v}_{in,y}  \delta{v}_{in,z}  \del ↪

↪ ta{\theta}_{in,x}  \delta{\theta}_{in,y}  \delta{\theta}_{in,z}])

### Kinematics

\begin{align*}
\delta p_i &\leftarrow \delta p_i + \delta v_i \Delta t \\
\text{Discrete Case} \qquad \qquad \delta v_i &\leftarrow \delta v_i + \delta v_{in} \\
\delta \theta_i &\leftarrow \delta \theta_i + \delta \theta_{in}\\
\end{align*}

In [264]:
f_delta_p_i = delta_p_i + delta_v_i * dt
f_delta_v_i = delta_v_i + delta_v_i_n
f_delta_theta_i = delta_theta_i + delta_theta_i_n

f_error_state_speaker = sym.Matrix.vstack(f_delta_p_i, f_delta_v_i, f_delta_theta_i)
# f_error_state_speaker

## Combined State

\begin{align*}
\delta x = \begin{bmatrix} \delta p & \delta v & \delta \theta & \delta a_b & \delta \omega_b & \delta p_i & \delta v_i & \delta \theta_i & \delta p_{i + 1} & \delta v_{i + 1} & \delta \theta_{i + 1} & \cdots \end{bmatrix}^\top \\
\end{align*}

In [265]:
delta_x = sym.Matrix.vstack(delta_x_IMU, delta_x_speaker)
delta_x_n = sym.Matrix.vstack(delta_x_n_IMU, delta_x_n_speaker)
f_error_state = sym.Matrix.vstack(f_error_state_IMU, f_error_state_speaker)

f_error_state

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢         ⎛                                                                    ↪
⎢\Delta t⋅⎝\delta{\theta}ₓ⋅((-\tilde{a}_y + a_{b,y})⋅(-2⋅q_w⋅q_y - 2⋅qₓ⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                ⎛                                                  ↪
⎢ \Delta t⋅⎝\delta{\theta}ₓ⋅⎝(-\tilde{a}_y + a_{b,y})⋅(2⋅q_w⋅qₓ - 2⋅q_y⋅q_z) + ↪
⎢                                                                              ↪
⎢          ⎛                

---
# Measurements

## IMU

$$
a^h = \mathbf{R}^{\{q\}}g + a_b
$$

## Face Detection

$$
\begin{align}
q_i^h &= (q \otimes \Delta q_{I \to c})^{-1} \otimes q_i \\
p_i^h &= \mathbf{R}^{\{\Delta q_{I \to c}\}}(\mathbf{R}^{\{q\}}(p_i - p) - \Delta p_{I \to c}) \\
\\
\text{with} \qquad\Delta p_{I \to c} &= \begin{bmatrix}
0 & -0.0013 & - 0.00662
\end{bmatrix}^\top \\
\Delta q_{I \to c} &= \begin{bmatrix}
\displaystyle \cos\left(-\tfrac{102°}{2}\right) & \displaystyle \sin\left(-\tfrac{102°}{2}\right) & 0 & 0
\end{bmatrix}^\top
\end{align}
$$

In [266]:
# IMU
a_h = R(q).T * sym.Matrix([0, 0, gravity]) + a_b

# face detection
q_h_i = R(IMU_cam_rotation) * (R(q) * (p_i - p) - IMU_cam_translation)
p_h_i = left_quat_matrix(quat_inv(left_quat_matrix(q) * IMU_cam_rotation)) * q_i

# combined
h = sym.Matrix.vstack(a_h, q_h_i, p_h_i)
h

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                           ⎛                                                  ↪
⎢- 2⋅\Delta{q}_w⋅\Delta{q}ₓ⋅⎝-\Delta{p}_z + (-pₓ + p_{i,x})⋅(-2⋅q_w⋅q_y + 2⋅qₓ ↪
⎢                           

---
# Jacobians for the Filter Equations

## State Transition Jacobian $F_{\delta x}$

In [267]:
%%script true
# TODO not up to date anymore
# TODO add all the speaker stuff
F_delta_x = f_error_state.jacobian(delta_x)

F_delta_x_n = sym.eye(f_error_state.shape[0])[:,3:]
# since we always just add the noise (except for IMU velocity), the following jacobian is just a modified identity matrix
# F_delta_x_n = f_error_state.jacobian(delta_x_n)

F_delta_x_n

## Measurement Jacobian $H$

$$
H = \frac{\partial h}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} = H_x X_{\delta x}
$$
The true state is just the nominal state with the error injected $x_t = x \oplus \delta x$ (special case for quaternions)

In [268]:
%%script true
#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :]

p_i_v_i_true = x_speaker[0:6, :] + delta_x_speaker[0:6, :]
q_i_true = left_quat_matrix(x_speaker[6:10, :]) * quat_from_axis_angle(delta_x_speaker[6:9, :])

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true, p_i_v_i_true, q_i_true)

H_x = h.jacobian(x)
X_delta_x = true_state.jacobian(delta_x)

H_delta_x = sym.simplify(H_x * X_delta_x)

# X_delta_x
# H_delta_x
# H_x

---
# Observability

## Weak Observability of Nominal State

$$
\begin{align}
\text{rank}\left( \begin{bmatrix}
\frac{\partial L_f^0 h(x)}{\partial x} \\
\frac{\partial L_f^1 h(x)}{\partial x} \\
\vdots \\
\frac{\partial L_f^{n-1} h(x)}{\partial x} \\
\end{bmatrix} \right) = 
\text{rank}\left(\begin{bmatrix}
H \\ H F \\ H F^2 \\ \vdots \\ HF^{n-1}
\end{bmatrix}\right) \stackrel{?}{=} n
\end{align}
$$

This however is only a proof of concept since it is way more computationally expensive than what I will be doing with the error state observability.

In [269]:
H = h.jacobian(x)
F = f_nominal_state.jacobian(x)

obs_matrix_nominal_state = sym.Matrix.vstack(H, H * F)
# obs_matrix_nominal_state = H

obs_matrix_nominal_state

⎡                                              0                               ↪
⎢                                                                              ↪
⎢                                              0                               ↪
⎢                                                                              ↪
⎢                                              0                               ↪
⎢                                                                              ↪
⎢                                          2        2                          ↪
⎢                                     2⋅q_y  + 2⋅q_z  - 1                      ↪
⎢                                                                              ↪
⎢                                                   ⎛                2⎞        ↪
⎢-2⋅\Delta{q}_w⋅\Delta{q}ₓ⋅(2⋅q_w⋅q_y - 2⋅qₓ⋅q_z) + ⎝1 - 2⋅\Delta{q}ₓ ⎠⋅(-2⋅q_ ↪
⎢                                                                              ↪
⎢                           

In [271]:
all_symbols = list(obs_matrix_nominal_state.free_symbols)
all_symbols

In [272]:
temp_substitution = subs_fixed_values
temp = obs_matrix_nominal_state.xreplace(temp_substitution)
temp

⎡                                                    0                         ↪
⎢                                                                              ↪
⎢                                                    0                         ↪
⎢                                                                              ↪
⎢                                                    0                         ↪
⎢                                                                              ↪
⎢                                                2        2                    ↪
⎢                                           2⋅q_y  + 2⋅q_z  - 1                ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢1.95629520146761⋅q_w⋅q_y + 0.415823381635519⋅q_w⋅q_z + 0.415823381635519⋅qₓ⋅q ↪
⎢                                                                              ↪
⎢                           

## Weak Observability of Error State

Since the above calculation takes a long time, I will use the second option of showing observability.

$$\mathcal{O} = \begin{bmatrix} H_{\delta x} \\ H_{\delta x}F_{\delta x} \\ H_{\delta x}F_{\delta x}^2 \\ \vdots \\ H_{\delta x}F_{\delta x}^{n-1} \end{bmatrix}$$

With

$$
\begin{align}
H_{\delta x} &= \frac{\partial h(x \oplus \delta x)}{\partial (x \oplus \delta x)} \cdot \frac{\partial (x \oplus \delta x)}{\partial \delta x} = \frac{\partial h}{\partial x_t} \frac{\partial x_t}{\partial \delta x} \\
F_{\delta x} &= \frac{\partial f_{\delta x}(\delta x)}{\partial \delta x}
\end{align}
$$

In an Error-State Kalman Filter (ESEKF), the nominal state $x$ absorbs the large non-linear trajectory, keeping the true error $\delta x$ close to zero. Because measurements arrive frequently, the error never grows large. Furthermore, after every measurement update, the estimated error is injected back into the nominal state and reset to zero ($\mathbb{E}[\delta x] = \mathbf{0}$). Thus, $\delta x = \mathbf{0}$ is the exact point around which the filter operates.

In [105]:
time_start = time.time()
n_states = delta_x.shape[0]
print(f"number of variables in state {n_states} (after {time.time() - time_start:.2f}s)")

#true state
p_v_true = x_IMU[0:6, :] + delta_x_IMU[0:6, :]
q_true = left_quat_matrix(x_IMU[6:10, :]) * quat_from_axis_angle(delta_x_IMU[6:9, :])
a_b_omega_b_true = x_IMU[10:16, :] + delta_x_IMU[9:15, :]
p_i_v_i_true = x_speaker[0:6, :] + delta_x_speaker[0:6, :]
q_i_true = left_quat_matrix(x_speaker[6:10, :]) * quat_from_axis_angle(delta_x_speaker[6:9, :])

true_state = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true, p_i_v_i_true, q_i_true)
print(f"true state constructed (after {time.time() - time_start:.2f}s)")

H = h.jacobian(x) * true_state.jacobian(delta_x)
print(f"H matrix constructed (after {time.time() - time_start:.2f}s)")
F = f_error_state.jacobian(delta_x)
# F = f_nominal_state.jacobian(x) * true_state.jacobian(delta_x)
print(f"F matrix constructed (after {time.time() - time_start:.2f}s)")

# Substitute zero error (see markdown explanation)
H = H.subs({var: 0 for var in delta_x})
F = F.subs({var: 0 for var in delta_x})
print(f"zero error substituted (after {time.time() - time_start:.2f}s)")

H = sym.simplify(H)
print(f"H matrix simplified (after {time.time() - time_start:.2f}s)")
F = sym.simplify(F)
print(f"F matrix simplified (after {time.time() - time_start:.2f}s)")

H.shape, F.shape

number of variables in state 24 (after 0.00s)
true state constructed (after 0.02s)
H matrix constructed (after 0.21s)
F matrix constructed (after 0.35s)
zero error substituted (after 3.40s)
H matrix simplified (after 13.19s)
F matrix simplified (after 15.12s)


Since we can often times determine rank after just a few entries of the above stated observability matrix, I will do it by checking after each iteration.

In [ ]:
%%script true 
# old implementation with unfinished nullspace calculation, keeping this block for reference
calculation_time = time.time()

# Map all symbols to random numbers ONCE outside the loop for numerical consistency
all_symbols = list(H.free_symbols.union(F.free_symbols))
np.random.seed(7)
symbol_dict = dict(zip(all_symbols, np.random.randn(len(all_symbols))))

observability_matrix = H
HF_k = H
previous_rank = 0
null_vectors = None

# Loop through k = 0 (H) up to n_states - 1 (Cayley-Hamilton cap)
for k in range(n_states):
    loop_start = time.time()

    print(
        f"\nBlock H{('F^' + str(k)) if k != 0 else ''} calculated | Shape = {observability_matrix.shape} | Evaluating rank..."
    )

    # 1. Substitute numerical values and convert to float64 NumPy array
    O_sym_eval = observability_matrix.xreplace(symbol_dict)
    O_num = np.array(O_sym_eval.tolist(), dtype=np.float64)

    # 2. Single-pass SVD (computes singular values S and right singular vectors Vh)
    U, S, Vh = np.linalg.svd(O_num, full_matrices=False)

    # 3. Calculate rank from S using exact NumPy numerical tolerance
    tol = max(O_num.shape) * np.finfo(O_num.dtype).eps * S[0]
    rank = int(np.sum(S > tol))

    # 4. Extract nullspace basis vectors directly from Vh (zero extra computation)
    null_vectors = Vh[rank:].T  # Shape: (n_states, n_unobservable)

    elapsed = time.time() - calculation_time
    loop_time = time.time() - loop_start

    print(f"System Numerical Rank: {rank} / {n_states}")
    print(f"evaluated in {loop_time:.1f}s (Total: {elapsed:.1f}s)")

    if rank == n_states:
        print("\nSystem Status: FULLY OBSERVABLE")
        null_vectors = np.empty((n_states, 0))
        break

    # Stop condition: Rank failed to grow after adding a new block
    if k > 0 and rank == previous_rank:
        print(
            f"\nRank saturated at {rank}/{n_states} with block HF^{k}. No further rank increase possible. Stopping."
        )
        break

    previous_rank = rank

    # Build bigger matrix block by block
    HF_k = HF_k * F
    observability_matrix = sym.Matrix.vstack(observability_matrix, HF_k)

if rank < n_states:
    n_unobs = n_states - rank
    print(
        f"\nSystem Status: UN-OBSERVABLE ({n_unobs} unobservable state{'s' if n_unobs > 1 else ''})"
    )
    # Zero out floating-point noise (< 1e-10) and round for clean display
    clean_null = np.where(np.abs(null_vectors) < 0.06, 0, np.round(null_vectors, 4))
    null_sym = sym.Matrix(clean_null)

    print("\nNullspace Basis Matrix (Columns = Unobservable State Directions):")
    display(Math(sym.latex(x) + sym.latex(null_sym)))

    # print(f"Nullspace Basis Matrix Shape: {null_vectors.shape}")


Block H calculated | Shape = (10, 24) | Evaluating rank...
System Numerical Rank: 9 / 24
evaluated in 0.1s (Total: 0.1s)

Block HF^1 calculated | Shape = (20, 24) | Evaluating rank...
System Numerical Rank: 15 / 24
evaluated in 0.1s (Total: 0.4s)

Block HF^2 calculated | Shape = (30, 24) | Evaluating rank...
System Numerical Rank: 17 / 24
evaluated in 0.4s (Total: 1.0s)

Block HF^3 calculated | Shape = (40, 24) | Evaluating rank...
System Numerical Rank: 17 / 24
evaluated in 0.9s (Total: 2.2s)

Rank saturated at 17/24 with block HF^3. No further rank increase possible. Stopping.

System Status: UN-OBSERVABLE (7 unobservable states)

Nullspace Basis Matrix (Columns = Unobservable State Directions):


<IPython.core.display.Math object>

### Rank and Nullspace

In [ ]:
calculation_time = time.time()

# Map all symbols to random numbers ONCE outside the loop for numerical consistency
all_symbols = list(H.free_symbols.union(F.free_symbols))
np.random.seed(7)
symbol_dict = dict(zip(all_symbols, np.random.randn(len(all_symbols))))

observability_matrix = H
HF_k = H
previous_rank = 0
null_vectors = None

# Loop through k = 0 (H) up to n_states - 1 (Cayley-Hamilton cap)
for k in range(n_states):
    loop_start = time.time()

    # 1. Substitute numerical values and convert to float64 NumPy array
    O_sym_eval = observability_matrix.xreplace(symbol_dict)
    O_num = np.array(O_sym_eval.tolist(), dtype=np.float64)

    # 2. Single-pass SVD (computes singular values S and right singular vectors Vh)
    U, S, Vh = np.linalg.svd(O_num, full_matrices=False)

    # 3. Calculate rank from S using exact NumPy numerical tolerance
    tol = max(O_num.shape) * np.finfo(O_num.dtype).eps * S[0]
    rank = int(np.sum(S > tol))

    # 4. Extract nullspace basis vectors directly from Vh (zero extra computation)
    null_vectors = Vh[rank:].T  # Shape: (n_states, n_unobservable)

    elapsed = time.time() - calculation_time
    loop_time = time.time() - loop_start

    print(
        f"\nBlock H{('F^' + str(k)) if k != 0 else ''} calculated | Shape = {observability_matrix.shape} | Rank: {rank} / {n_states}"
    )
    # print(f"System Numerical Rank: {rank} / {n_states}")
    print(f"evaluated in {loop_time:.1f}s (Total: {elapsed:.1f}s)")

    if rank == n_states:
        print("\nSystem Status: FULLY OBSERVABLE")
        null_vectors = np.empty((n_states, 0))
        break

    # Stop condition: Rank failed to grow after adding a new block
    if k > 0 and rank == previous_rank:
        print(
            f"\nRank saturated at {rank}/{n_states} with block HF^{k}. No further rank increase possible. Stopping loop."
        )
        break

    previous_rank = rank

    # Build bigger matrix block by block
    HF_k = HF_k * F
    observability_matrix = sym.Matrix.vstack(observability_matrix, HF_k)


Block H calculated | Shape = (10, 24) | Rank: 9 / 24
evaluated in 0.0s (Total: 0.0s)

Block HF^1 calculated | Shape = (20, 24) | Rank: 15 / 24
evaluated in 0.1s (Total: 0.2s)

Block HF^2 calculated | Shape = (30, 24) | Rank: 17 / 24
evaluated in 0.5s (Total: 0.7s)

Block HF^3 calculated | Shape = (40, 24) | Rank: 17 / 24
evaluated in 1.0s (Total: 1.9s)

Rank saturated at 17/24 with block HF^3. No further rank increase possible. Stopping.


In [115]:
if rank < n_states:
    n_unobs = n_states - rank
    print(
        f"\nSystem Status: UN-OBSERVABLE ({n_unobs} unobservable state{'s' if n_unobs > 1 else ''})"
    )
   
    clean_null = np.where(np.abs(null_vectors) < 0.08, 0.0, null_vectors)

    # 2. Apply RREF to un-mix the SVD coordinate rotation into canonical axes
    rref_mat, _ = sym.Matrix(clean_null.T).rref()
    canonical_null = np.array(rref_mat.T, dtype=float)

    # 3. Map state symbols onto the clean canonical matrix
    rows, cols = canonical_null.shape
    symbolic_clean = []
    for i in range(rows):
        state_var = delta_x[i]
        row_entries = []
        for j in range(cols):
            val = canonical_null[i, j]
            if abs(val) < 1e-5:
                row_entries.append(0)
            else:
                row_entries.append(state_var if val > 0 else -state_var)
        symbolic_clean.append(row_entries)

    print(f"Nullspace Basis Matrix Shape: {null_vectors.shape}")
    display(sym.Matrix(symbolic_clean))


System Status: UN-OBSERVABLE (7 unobservable states)
Nullspace Basis Matrix Shape: (24, 7)


⎡     \delta{p}ₓ               0                 0                0            ↪
⎢                                                                              ↪
⎢          0              \delta{p}_y            0                0            ↪
⎢                                                                              ↪
⎢          0                   0            \delta{p}_z           0            ↪
⎢                                                                              ↪
⎢          0                   0                 0           \delta{v}ₓ        ↪
⎢                                                                              ↪
⎢          0                   0                 0                0            ↪
⎢                                                                              ↪
⎢          0                   0                 0                0            ↪
⎢                                                                              ↪
⎢          0                

---
# debugging

In [39]:
# quaternion test
q_test = sym.Matrix(sym.symbols("q_w, q_x, q_y, q_z"))
p = sym.Matrix(sym.symbols("x, y, z"))

# axis angle test
result_b = quat_from_axis_angle(p, False)

# skew-symmetric matrix test
skew_symmetric_matrix(p)

# H matrix only for IMU
true_state_IMU = sym.Matrix.vstack(p_v_true, q_true, a_b_omega_b_true)

H_x_IMU = h_g.jacobian(x_IMU)
X_delta_x_IMU = true_state_IMU.jacobian(delta_x_IMU)

H_delta_x_IMU = H_x_IMU * X_delta_x_IMU
sym.simplify(H_delta_x_IMU)

⎡                                                          ⎛     2     2       ↪
⎢0  0  0  0  0  0                 0                  1.0⋅g⋅⎝- q_w  + qₓ  + q_y ↪
⎢                                                                              ↪
⎢                        ⎛   2     2      2      2⎞                            ↪
⎢0  0  0  0  0  0  1.0⋅g⋅⎝q_w  - qₓ  - q_y  + q_z ⎠                  0         ↪
⎢                                                                              ↪
⎣0  0  0  0  0  0     2.0⋅g⋅(-q_w⋅qₓ - q_y⋅q_z)          2.0⋅g⋅(-q_w⋅q_y + qₓ⋅ ↪

↪ 2      2⎞                                            ⎤
↪   - q_z ⎠  2.0⋅g⋅(q_w⋅qₓ + q_y⋅q_z)  1  0  0  0  0  0⎥
↪                                                      ⎥
↪                                                      ⎥
↪            2.0⋅g⋅(q_w⋅q_y - qₓ⋅q_z)  0  1  0  0  0  0⎥
↪                                                      ⎥
↪ q_z)                  0              0  0  1  0  0  0⎦